# Fraud Detection Robustness Benchmark: Colab T4 Research Run

This notebook runs the repository benchmark on Google Colab with an NVIDIA T4 GPU. It is organized as a research workflow rather than a classroom lab: environment checks, configuration traceability, staged execution, live progress, resumability, artifact validation, and a compact results audit.

The default path runs a fast smoke test first. After that passes, set `RUN_MAIN = True` in the configuration cell to run the report-facing v3 benchmark.

## Runtime Requirements

Use `Runtime > Change runtime type > T4 GPU` before running this notebook.

The notebook assumes Colab Linux commands, so it uses `python -m benchmark.run`, not Windows `py -m benchmark.run`. Outputs should go to Google Drive for long runs because Colab sessions can disconnect.

In [ ]:
# Quick GPU check. This should print an NVIDIA T4 in a correct Colab runtime.
!nvidia-smi

## Install Runtime Stack

Run this install cell once in a fresh Colab runtime, then restart the runtime when the next cell asks you to. The pinned stack avoids common DGL/PyTorch compatibility issues on Colab.

Set `RUN_INSTALL = True` only when the environment is not already prepared.

In [ ]:
RUN_INSTALL = False

if RUN_INSTALL:
    import subprocess
    import sys
    print(sys.version)
    assert sys.version_info[:2] == (3, 11), "This notebook expects Colab Python 3.11."

    def pip_cmd(args):
        subprocess.run([sys.executable, "-m", "pip"] + list(args), check=True)

    pip_cmd(["uninstall", "-y", "dgl", "torch", "torchdata", "torchvision", "torchaudio"])
    pip_cmd(["install", "-q", "numpy<2", "scipy", "sympy", "PyYAML", "pydantic", "matplotlib", "tqdm", "scikit-learn", "pandas"])
    pip_cmd(["install", "-q", "torch==2.1.0", "torchvision==0.16.0", "torchaudio==2.1.0", "--index-url", "https://download.pytorch.org/whl/cu118"])
    pip_cmd(["install", "-q", "dgl==1.1.3+cu118", "-f", "https://data.dgl.ai/wheels/cu118/repo.html"])
else:
    print("Install skipped. Set RUN_INSTALL = True if this runtime is not prepared yet.")

In [ ]:
# Restart after installing. After restart, run from the next section.
if RUN_INSTALL:
    import os
    os.kill(os.getpid(), 9)

## Locate Repository And Set Output Paths

If the repository is not already present in `/content`, set `REPO_URL` to your GitHub URL and set `CLONE_IF_MISSING = True`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/<YOUR_USER_OR_ORG>/fraud-detection-robustness-benchmark.git"
PROJECT_DIR = Path("/content/fraud-detection-robustness-benchmark")
CLONE_IF_MISSING = False
USE_DRIVE = True

def looks_like_repo(path: Path) -> bool:
    return (path / "benchmark" / "run.py").exists() and (path / "configs").exists()

cwd = Path.cwd()
if looks_like_repo(cwd):
    PROJECT_DIR = cwd
elif looks_like_repo(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
elif CLONE_IF_MISSING:
    if "<YOUR_USER_OR_ORG>" in REPO_URL:
        raise RuntimeError("Set REPO_URL to your actual GitHub repository URL before cloning.")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    os.chdir(PROJECT_DIR)
else:
    raise RuntimeError("Repository not found. Upload it to Colab, cd into it, or enable cloning with REPO_URL.")

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RUN_ROOT = Path("/content/drive/MyDrive/fraud-benchmark-runs")
    except ModuleNotFoundError:
        print("google.colab is not available; using local runs/ output.")
        RUN_ROOT = PROJECT_DIR / "runs"
else:
    RUN_ROOT = PROJECT_DIR / "runs"

OUT_SMOKE = RUN_ROOT / "gfd_robustness_v3_smoke_colab"
OUT_MAIN = RUN_ROOT / "gfd_robustness_benchmark_v3_colab"
OUT_SMOKE.mkdir(parents=True, exist_ok=True)
OUT_MAIN.mkdir(parents=True, exist_ok=True)

os.environ["PROJECT_DIR"] = str(PROJECT_DIR)
os.environ["OUT_SMOKE"] = str(OUT_SMOKE)
os.environ["OUT_MAIN"] = str(OUT_MAIN)

print("Project:", PROJECT_DIR)
print("Smoke output:", OUT_SMOKE)
print("Main output:", OUT_MAIN)

## Research Run Controls

The benchmark is resumable through `results.csv`. Keep model stages chunked so a Colab disconnect does not waste completed work.

In [ ]:
CONFIG_SMOKE = "configs/exp_yelpchi_v3_fast.json"
CONFIG_MAIN = "configs/exp_yelpchi_v3.json"

# Run the smoke test first. Set RUN_MAIN = True only after the smoke test passes.
RUN_SMOKE = True
RUN_MAIN = False

DEVICE = "cuda"
MODEL_CHUNKS = ["mlp,sage", "pmp", "secgfd"]

# Keep 0 to use config/default values. Use small values for exploratory Colab runs.
MAX_EPOCHS = 0
PATIENCE = 0

# If True, reduce SEC-GFD cost for T4 feasibility. Disclose this if used for final results.
SECGFD_T4_SAFE = False

# Graph generation is not resumable in the same way as training. Leave false unless you want to rebuild cached graphs.
FORCE_REBUILD_GRAPHS = False

print({
    "RUN_SMOKE": RUN_SMOKE,
    "RUN_MAIN": RUN_MAIN,
    "DEVICE": DEVICE,
    "MODEL_CHUNKS": MODEL_CHUNKS,
    "MAX_EPOCHS": MAX_EPOCHS,
    "PATIENCE": PATIENCE,
    "SECGFD_T4_SAFE": SECGFD_T4_SAFE,
})

## Environment Validation

This records enough environment information to explain later results and catches common GPU/DGL mistakes before long runs start.

In [ ]:
import json
import platform
import time
import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import dgl

print("python:", sys.version)
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("dgl:", dgl.__version__)
print("numpy:", np.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
    print("cuda capability:", torch.cuda.get_device_capability(0))
    test_graph = dgl.rand_graph(10, 20).to("cuda")
    print("DGL CUDA test graph nodes:", test_graph.num_nodes())
elif DEVICE == "cuda":
    raise RuntimeError("DEVICE is cuda, but torch.cuda.is_available() is false.")

## Config Inspection

This cell prints the scenario surface, seed counts, and expected run scale before any training starts.

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def summarize_config(config_path):
    cfg = load_json(config_path)
    graph_seeds = cfg.get("seeds", {}).get("graph_seeds", cfg.get("seeds", {}).get("training_seeds", []))
    training_seeds = cfg.get("seeds", {}).get("training_seeds", [])
    models = [m.get("model_id") for m in cfg.get("models", [])]
    scenario_rows = []
    for s in cfg.get("scenarios", []):
        scenario_rows.append({
            "scenario_id": s.get("scenario_id"),
            "oracle": bool(s.get("oracle_labels", False)),
            "graph_view_mode": s.get("graph_view_mode", "canonical"),
            "severity_values": s.get("severity_values", []),
        })
    n_scenario_variants = sum(len(s["severity_values"]) * len(graph_seeds) for s in scenario_rows)
    n_clean = len(cfg.get("datasets", [])) * len(cfg.get("data_splits", []))
    n_variants = n_clean + n_scenario_variants * n_clean
    print("config:", config_path)
    print("sha256:", sha256_file(config_path))
    print("experiment:", cfg.get("experiment_name"))
    print("models:", models)
    print("graph seeds:", graph_seeds)
    print("training seeds:", training_seeds)
    print("expected variant rows:", n_variants)
    print("train_on_variant run count:", n_variants * len(training_seeds) * len(models))
    display(pd.DataFrame(scenario_rows))
    return cfg

cfg_smoke = summarize_config(CONFIG_SMOKE)
cfg_main = summarize_config(CONFIG_MAIN)

## Live Benchmark Runner

The helper below starts each stage with an immediate visible status panel, streams repository progress lines live, and parses `[x/y]` progress from the benchmark's own preflight/progress output when available.

In [ ]:
import html
import re
import shlex
from IPython.display import HTML, display

def progress_html(label, phase, done=None, total=None, elapsed=0.0, extra=""):
    pct = 0.0
    if done is not None and total:
        pct = max(0.0, min(100.0, 100.0 * float(done) / float(total)))
    width = f"{pct:.1f}%" if total else "100%"
    bar_color = "#2f855a" if phase == "completed" else "#3182ce"
    if phase == "failed":
        bar_color = "#c53030"
    if done is not None and total:
        progress_text = f"{done}/{total} ({pct:.1f}%)"
    elif phase == "completed":
        progress_text = "done"
    elif phase == "failed":
        progress_text = "failed"
    else:
        progress_text = "running"
    return f"""
    <div style='border:1px solid #cbd5e0; border-radius:6px; padding:10px; font-family:monospace;'>
      <div><b>{html.escape(label)}</b> - {html.escape(phase)} - {progress_text} - elapsed {elapsed:.1f}s</div>
      <div style='height:10px; background:#edf2f7; margin-top:6px; border-radius:5px; overflow:hidden;'>
        <div style='height:10px; width:{width}; background:{bar_color};'></div>
      </div>
      <div style='margin-top:6px; color:#4a5568'>{html.escape(str(extra))}</div>
    </div>
    """

def run_benchmark(args, label, cwd=PROJECT_DIR):
    cmd = [sys.executable, "-m", "benchmark.run"] + [str(a) for a in args]
    start = time.time()
    panel = display(HTML(progress_html(label, "starting", elapsed=0.0, extra=" ".join(shlex.quote(x) for x in cmd))), display_id=True)
    print("\n===", label, "===")
    print("Started:", datetime.now(timezone.utc).isoformat())
    print("Command:", " ".join(shlex.quote(x) for x in cmd))
    print()

    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    last_done = None
    last_total = None
    last_line = "process started"
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
        stripped = line.strip()
        if stripped:
            last_line = stripped
        m = re.search(r"\[(\d+)/(\d+)\]", line)
        if m:
            last_done = int(m.group(1))
            last_total = int(m.group(2))
            panel.update(HTML(progress_html(label, "running", last_done, last_total, time.time() - start, stripped)))
        elif stripped.startswith("[preflight]") or stripped.startswith("[graphs]") or stripped.startswith("[run]"):
            panel.update(HTML(progress_html(label, "running", last_done, last_total, time.time() - start, stripped)))

    code = proc.wait()
    elapsed = time.time() - start
    if code != 0:
        panel.update(HTML(progress_html(label, "failed", last_done, last_total, elapsed, last_line)))
        raise RuntimeError(f"{label} failed with exit code {code}")
    panel.update(HTML(progress_html(label, "completed", last_total, last_total, elapsed, last_line)))
    print(f"\nCompleted {label} in {elapsed:.1f}s")

## Run Manifest

The manifest captures code/config/environment state before execution. It makes the Colab run easier to audit later.

In [ ]:
def git_value(args):
    try:
        return subprocess.check_output(["git"] + args, cwd=str(PROJECT_DIR), text=True).strip()
    except Exception as e:
        return f"unavailable: {e}"

def write_manifest(out_dir: Path, config_path: str, label: str):
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest = {
        "label": label,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "project_dir": str(PROJECT_DIR),
        "config_path": config_path,
        "config_sha256": sha256_file(config_path),
        "git_commit": git_value(["rev-parse", "HEAD"]),
        "git_status_short": git_value(["status", "--short"]),
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "dgl": dgl.__version__,
        "numpy": np.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "device_requested": DEVICE,
        "max_epochs_override": MAX_EPOCHS,
        "patience_override": PATIENCE,
        "secgfd_t4_safe": SECGFD_T4_SAFE,
    }
    path = out_dir / f"colab_manifest_{label}.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("wrote", path)
    return manifest

if RUN_SMOKE:
    manifest_smoke = write_manifest(OUT_SMOKE, CONFIG_SMOKE, "smoke")
if RUN_MAIN:
    manifest_main = write_manifest(OUT_MAIN, CONFIG_MAIN, "main")

## Fast Smoke Test

This should finish much faster than the main benchmark. It validates DGL dataset loading, graph caching, training, result writing, and plotting.

In [ ]:
if RUN_SMOKE:
    run_benchmark([
        "--config", CONFIG_SMOKE,
        "--stage", "graphs",
        "--out", OUT_SMOKE,
        "--force",
    ], "smoke: graph generation")

    run_benchmark([
        "--config", CONFIG_SMOKE,
        "--stage", "baselines",
        "--out", OUT_SMOKE,
        "--device", DEVICE,
        "--max-epochs", 5,
        "--patience", 2,
        "--force",
    ], "smoke: baseline training")

    run_benchmark([
        "--config", CONFIG_SMOKE,
        "--stage", "plots",
        "--out", OUT_SMOKE,
    ], "smoke: plots")
else:
    print("Smoke test skipped.")

## Main V3 Graph Cache

Graph generation is deterministic by graph seed. The cell skips regeneration if `graph_variants.csv` already exists unless `FORCE_REBUILD_GRAPHS = True`.

In [ ]:
if RUN_MAIN:
    variants_path = OUT_MAIN / "graph_variants.csv"
    if FORCE_REBUILD_GRAPHS or not variants_path.exists():
        args = [
            "--config", CONFIG_MAIN,
            "--stage", "graphs",
            "--out", OUT_MAIN,
        ]
        if FORCE_REBUILD_GRAPHS:
            args.append("--force")
        run_benchmark(args, "main: graph generation")
    else:
        print("Using existing graph cache:", variants_path)
else:
    print("Main benchmark disabled. Set RUN_MAIN = True in the controls cell after smoke passes.")

## Main V3 Training: `train_on_variant`

This protocol retrains each selected model on each graph variant. Runs are chunked by model family for Colab stability.

In [ ]:
def training_args_for_models(models: str):
    args = ["--device", DEVICE]
    if MAX_EPOCHS:
        args += ["--max-epochs", str(MAX_EPOCHS)]
    if PATIENCE:
        args += ["--patience", str(PATIENCE)]
    if SECGFD_T4_SAFE and "secgfd" in models.split(","):
        args += ["--secgfd-hid-dim", "16", "--secgfd-order-d", "1", "--secgfd-high-order", "1"]
    return args

if RUN_MAIN:
    for models in MODEL_CHUNKS:
        run_benchmark([
            "--config", CONFIG_MAIN,
            "--stage", "matrix",
            "--models", models,
            "--out", OUT_MAIN,
        ] + training_args_for_models(models), f"main train_on_variant: {models}")
else:
    print("Main train_on_variant disabled.")

## Main V3 Training: `train_clean_eval_all`

This protocol trains on the clean graph once per split/model/seed, then evaluates the trained artifact on all variants. It answers a different robustness question from `train_on_variant`, so keep the protocols separate in analysis.

In [ ]:
if RUN_MAIN:
    for models in MODEL_CHUNKS:
        run_benchmark([
            "--config", CONFIG_MAIN,
            "--stage", "matrix",
            "--protocol", "train_clean_eval_all",
            "--models", models,
            "--out", OUT_MAIN,
        ] + training_args_for_models(models), f"main train_clean_eval_all: {models}")
else:
    print("Main shift protocol disabled.")

## Generate Report Artifacts

The plots stage creates summary CSVs, missing/error diagnostics, performance-audit joins, and PNG curves.

In [ ]:
if RUN_MAIN:
    run_benchmark([
        "--config", CONFIG_MAIN,
        "--stage", "plots",
        "--out", OUT_MAIN,
        "--ci",
    ], "main: plots and summaries")
else:
    print("Main plot generation disabled.")

## Artifact And Completeness Audit

Run this after smoke or main stages. It checks required files, status counts, error rows, and missing-run diagnostics.

In [ ]:
def audit_outputs(out_dir: Path):
    print("Auditing:", out_dir)
    expected = [
        "config.json",
        "graph_variants.csv",
        "variant_audit.csv",
        "results.csv",
        "plots/summary_curves.csv",
        "plots/performance_drop_max_stress.csv",
        "plots/robustness_scores.csv",
        "plots/audit_curves.csv",
        "plots/performance_audit_join.csv",
        "plots/missing_or_error_runs.csv",
    ]
    file_rows = []
    for rel in expected:
        path = out_dir / rel
        file_rows.append({"artifact": rel, "exists": path.exists(), "size_bytes": path.stat().st_size if path.exists() else None})
    display(pd.DataFrame(file_rows))

    results_path = out_dir / "results.csv"
    if results_path.exists():
        results = pd.read_csv(results_path)
        print("results rows:", len(results))
        display(results.groupby(["protocol", "model_id", "status"]).size().reset_index(name="n"))
        err = results[results["status"].astype(str).str.lower().eq("error")]
        if len(err):
            display(err[["protocol", "model_id", "scenario_id", "severity", "graph_seed", "training_seed", "error"]].head(20))

    missing_path = out_dir / "plots" / "missing_or_error_runs.csv"
    if missing_path.exists():
        missing = pd.read_csv(missing_path)
        print("missing/error diagnostic rows:", len(missing))
        display(missing.head(20))

audit_target = OUT_MAIN if RUN_MAIN else OUT_SMOKE
audit_outputs(audit_target)

## Inspect Summary Tables And Curves

Use this for quick sanity checks before writing report conclusions. Do not average across protocols.

In [ ]:
from IPython.display import Image

def inspect_summaries(out_dir: Path):
    plots_dir = out_dir / "plots"
    summary_path = plots_dir / "summary_curves.csv"
    audit_path = plots_dir / "audit_curves.csv"
    if summary_path.exists():
        summary = pd.read_csv(summary_path)
        display(summary.head(20))
        display(summary.groupby(["protocol", "model_id", "metric"]).size().reset_index(name="rows"))
    if audit_path.exists():
        audit = pd.read_csv(audit_path)
        display(audit.head(20))
    pngs = sorted(plots_dir.rglob("*.png"))[:8]
    print("showing", len(pngs), "plot files")
    for path in pngs:
        print(path.relative_to(out_dir))
        display(Image(filename=str(path)))

inspect_summaries(audit_target)

## Retry Error Rows

If a model stage produces `status=error` rows because of a transient Colab issue, retry the affected chunk with `--retry-errors`. Keep this cell disabled unless needed.

In [ ]:
RUN_RETRY = False
RETRY_MODELS = "secgfd"
RETRY_PROTOCOL = "train_on_variant"  # or "train_clean_eval_all"

if RUN_RETRY:
    retry_args = [
        "--config", CONFIG_MAIN,
        "--stage", "matrix",
        "--models", RETRY_MODELS,
        "--out", OUT_MAIN,
        "--retry-errors",
    ] + training_args_for_models(RETRY_MODELS)
    if RETRY_PROTOCOL == "train_clean_eval_all":
        retry_args += ["--protocol", "train_clean_eval_all"]
    run_benchmark(retry_args, f"retry errors: {RETRY_MODELS} {RETRY_PROTOCOL}")
else:
    print("Retry disabled.")

## Archive Outputs

Use this after the final plots stage. If `OUT_MAIN` is already on Google Drive, the archive is optional.

In [ ]:
CREATE_ARCHIVE = False

if CREATE_ARCHIVE:
    import shutil
    archive_base = Path("/content") / audit_target.name
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=str(audit_target))
    print("archive:", archive_path)
    try:
        from google.colab import files
        files.download(archive_path)
    except ModuleNotFoundError:
        print("Not in Colab; archive created locally.")
else:
    print("Archive disabled.")

## Interpretation Notes

- `results.csv` is the unified performance table.
- `variant_audit.csv` records what each perturbation actually changed.
- `plots/performance_audit_join.csv` joins performance rows with perturbation audit evidence.
- Keep `train_on_variant` and `train_clean_eval_all` separate in all conclusions.
- Disclose oracle scenarios: `heterophily_rewire_oracle`, `camouflage_feature_oracle`, and `camouflage_relation_oracle` use labels during perturbation construction.
- Treat severity as family-specific. Use realized audit metrics when comparing stress strength.